# Sistema de Detección de Intrusiones en Redes usando Machine Learning sobre el Dataset NSL-KDD

**Universidad Internacional del Ecuador — UIDE**
Ingeniería en Sistemas | Big Data

| | |
|---|---|
| **Integrante** | Andres Quisilema |
| **Entregable** | 3 — Implementación, Resultados y Validación de Hipótesis |
| **Fecha** | Junio 2026 |

---

### Descripción general del proyecto

Este proyecto busca construir un sistema de detección de intrusiones en redes usando Machine Learning sobre el dataset NSL-KDD. El objetivo es clasificar el tráfico de red como normal o malicioso usando tres modelos supervisados: Regresión Logística, Random Forest y KNN. Este notebook cubre el ciclo completo: desde la extracción de datos hasta la validación de las hipótesis planteadas.


---
## 1. Instalación e Importación de Librerías

In [ ]:
!pip install kagglehub -q

import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')
sns.set_palette("Set2")
print("Librerías importadas correctamente.")


---
## 2. Extracción de Datos — API de Kaggle

Los datos se descargan directamente desde la API oficial de Kaggle usando `kagglehub`.
Esta librería se conecta al servidor de Kaggle, valida las credenciales mediante un token
personal y descarga los archivos programáticamente. Eso cumple con el concepto de consumo
de API pública: hay autenticación real, una petición al servidor y una respuesta con los datos.


In [ ]:
# Configuración del token de la API de Kaggle
# Obtenerlo en: kaggle.com → Settings → API → Create New Token
import os
os.environ['KAGGLE_API_TOKEN'] = 'pega_tu_token_aqui'
print("Token configurado.")


In [ ]:
# Descarga del dataset NSL-KDD via API de Kaggle
print("Conectando con la API de Kaggle...")
path = kagglehub.dataset_download("hassan06/nslkdd")
print("Dataset descargado en:", path)
print("Archivos disponibles:", os.listdir(path))


In [ ]:
# Nombres oficiales de las 43 columnas del dataset NSL-KDD
columnas = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes','land',
    'wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
    'num_shells','num_access_files','num_outbound_cmds','is_host_login',
    'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
    'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty_level'
]

# Carga de los archivos de entrenamiento y prueba
df_train = pd.read_csv(os.path.join(path, 'KDDTrain+.txt'), header=None, names=columnas)
df_test  = pd.read_csv(os.path.join(path, 'KDDTest+.txt'),  header=None, names=columnas)

print(f"Registros de entrenamiento : {len(df_train):,}")
print(f"Registros de prueba        : {len(df_test):,}")
print(f"Variables por registro     : {df_train.shape[1]}")
print()
print("Variables obtenidas:")
print(list(df_train.columns))
print()
print("Vista previa del dataset:")
df_train.head(3)


---
## 3. Limpieza y Preprocesamiento

Antes de entrenar cualquier modelo, los datos necesitan estar limpios y en un formato
que los algoritmos puedan procesar. Esta sección cubre todos los pasos necesarios.


### 3.1 Revisión de valores nulos y duplicados

In [ ]:
# Verificación de valores nulos
print("=== VALORES NULOS ===")
print(f"Entrenamiento : {df_train.isnull().sum().sum()}")
print(f"Prueba        : {df_test.isnull().sum().sum()}")

# Verificación de duplicados
print()
print("=== DUPLICADOS ===")
dup_train = df_train.duplicated().sum()
print(f"Entrenamiento : {dup_train}")
if dup_train > 0:
    df_train = df_train.drop_duplicates()
    print(f"  → Eliminados. Registros restantes: {len(df_train):,}")
else:
    print("  → No hay duplicados.")


### 3.2 Creación de la etiqueta binaria

In [ ]:
# Transformamos la etiqueta original a clasificación binaria
# 0 = tráfico normal  |  1 = cualquier tipo de ataque
df_train['label_binario'] = df_train['label'].apply(lambda x: 0 if x == 'normal' else 1)
df_test['label_binario']  = df_test['label'].apply(lambda x: 0 if x == 'normal' else 1)

conteo = df_train['label_binario'].value_counts()
print("Distribución de clases en entrenamiento:")
print(f"  Normal  (0): {conteo[0]:,}  ({conteo[0]/len(df_train)*100:.1f}%)")
print(f"  Ataque  (1): {conteo[1]:,}  ({conteo[1]/len(df_train)*100:.1f}%)")


### 3.3 Encoding de variables categóricas

In [ ]:
# Las columnas protocol_type, service y flag contienen texto
# Los modelos de ML solo trabajan con números, así que las convertimos
le = LabelEncoder()
for col in ['protocol_type', 'service', 'flag']:
    le.fit(pd.concat([df_train[col], df_test[col]]))
    df_train[col] = le.transform(df_train[col])
    df_test[col]  = le.transform(df_test[col])
    print(f"  {col}: codificado correctamente")

print()
print("Encoding completado.")


### 3.4 Estandarización y separación de features

In [ ]:
# Separamos las features de la etiqueta objetivo
features = [c for c in df_train.columns if c not in ['label','label_binario','difficulty_level']]

X_train = df_train[features].copy()
y_train = df_train['label_binario'].copy()
X_test  = df_test[features].copy()
y_test  = df_test['label_binario'].copy()

# StandardScaler lleva todas las variables a la misma escala (media=0, std=1)
# Esto es especialmente importante para KNN, que trabaja con distancias
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"X_train: {X_train_scaled.shape}")
print(f"X_test : {X_test_scaled.shape}")
print(f"Features utilizadas: {len(features)}")


### 3.5 Exploración visual del dataset

In [ ]:
# Gráfico 1: Distribución de clases
conteo = df_train['label_binario'].value_counts()

plt.figure(figsize=(6, 4))
plt.bar(['Normal', 'Ataque'], conteo.values,
        color=['#2ecc71', '#e74c3c'], width=0.5, edgecolor='white')
plt.title('Distribución de clases — Dataset de entrenamiento')
plt.ylabel('Cantidad de registros')
for i, v in enumerate(conteo.values):
    plt.text(i, v + 500, f'{v:,}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()


### 3.6 Matriz de Correlación

In [ ]:
# Seleccionamos las 8 features más representativas para la matriz
# Con 41 variables el gráfico sería ilegible, estas cubren los distintos
# tipos de información: bytes, errores, sesiones y patrones de conexión

features_corr = [
    'src_bytes', 'dst_bytes', 'num_failed_logins',
    'logged_in', 'serror_rate', 'same_srv_rate',
    'count', 'label_binario'
]

traduccion = {
    'src_bytes'        : 'Bytes enviados',
    'dst_bytes'        : 'Bytes recibidos',
    'num_failed_logins': 'Intentos fallidos',
    'logged_in'        : 'Sesion iniciada',
    'serror_rate'      : 'Tasa de error SYN',
    'same_srv_rate'    : 'Tasa mismo servicio',
    'count'            : 'Conexiones recientes',
    'label_binario'    : 'Etiqueta (0=Normal 1=Ataque)',
}

correlacion = df_train[features_corr].corr()
correlacion = correlacion.rename(index=traduccion, columns=traduccion)

plt.figure(figsize=(11, 8))
sns.heatmap(
    correlacion,
    annot=True, fmt='.2f',
    cmap='RdYlGn', center=0,
    linewidths=0.5, annot_kws={'size': 9}
)
plt.title('Matriz de correlación — Features clave vs etiqueta binaria', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

# Ranking de correlación con la etiqueta
print("Correlación de cada feature con la etiqueta (0=Normal / 1=Ataque):")
print(correlacion['Etiqueta (0=Normal 1=Ataque)']
      .drop('Etiqueta (0=Normal 1=Ataque)')
      .sort_values(ascending=False)
      .to_string())


---
## 4. Hipótesis del Proyecto

Antes de entrenar los modelos se plantearon tres hipótesis basadas en la exploración
del dataset y en la lógica de cómo funcionan los ataques de red.

---

**Hipótesis 1:** El tráfico de ataque tiene valores numéricos notablemente distintos al tráfico normal en variables como `src_bytes`, `duration` y `serror_rate`, lo que debería permitir a los modelos separar las dos clases con buena precisión.

**Hipótesis 2:** El tipo de protocolo y el servicio de red van a ser de las variables más importantes para detectar ataques, porque ciertos tipos de ataque siempre usan los mismos protocolos o apuntan a los mismos servicios.

**Hipótesis 3:** Un modelo simple como Regresión Logística ya va a dar resultados aceptables, pero Random Forest y KNN van a mejorar la detección de los ataques menos frecuentes como U2R y R2L.


---
## 5. Implementación y Resultados de los Modelos

### Variables del problema

- **Variables de entrada (features):** 41 atributos de tráfico de red preprocesados mediante estandarización con `StandardScaler`. Incluyen variables numéricas como `src_bytes`, `duration`, `count`, y variables categóricas codificadas como `protocol_type`, `service` y `flag`.
- **Variable objetivo (target):** `label_binario` — 0 para tráfico normal, 1 para cualquier tipo de ataque.
- **Tipo de problema:** Clasificación binaria supervisada.


### Modelo 1 — Regresión Logística

**Justificación técnica:** Es el modelo más simple de los tres y cumple la función de línea base o *baseline*. Si los modelos más complejos no lo superan, hay un problema en el proceso. Sus coeficientes son directamente interpretables y el costo computacional es mínimo. Para un problema donde el tráfico normal y el de ataque tienen diferencias numéricas claras, debería dar resultados aceptables sin mucho ajuste.

**Ventajas:** Alta interpretabilidad, entrenamiento rápido, bajo consumo de recursos.

**Limitaciones:** Asume relaciones lineales entre las variables y la etiqueta. Ante patrones de ataque complejos y no lineales, su capacidad de detección se ve limitada, especialmente en categorías poco frecuentes como U2R y R2L.


### Modelo 2 — Random Forest

**Justificación técnica:** Conjunto de árboles de decisión que votan por mayoría. Maneja bien variables mixtas, no se ve afectado por variables irrelevantes y entrega un ranking de importancia de features que permite verificar la Hipótesis 2. Es el modelo con mayor rendimiento esperado de los tres.

**Ventajas:** Robusto ante el sobreajuste, no requiere asumir distribución de los datos, entrega importancia de variables.

**Limitaciones:** Mayor tiempo de entrenamiento que los otros dos modelos. Menos interpretable a nivel de decisiones individuales que la Regresión Logística.


### Modelo 3 — K-Nearest Neighbors (KNN)

**Justificación técnica:** Clasifica una conexión buscando las K conexiones más similares en el dataset de entrenamiento y asignando la clase más frecuente entre ellas. Su enfoque es completamente distinto a los otros dos, lo que lo hace útil para comparar. Es sensible a la escala, por eso es importante haber aplicado `StandardScaler` antes.

**Ventajas:** No requiere fase de entrenamiento explícita, intuitivo de entender y explicar.

**Limitaciones:** Más lento en la fase de predicción con datasets grandes, ya que calcula distancias para cada nueva observación. El rendimiento depende del valor de K elegido.


### Entrenamiento y evaluación de los tres modelos

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Definición de los tres modelos
modelos = {
    'Regresion Logistica': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'KNN'                : KNeighborsClassifier(n_neighbors=5)
}

# Diccionario para guardar predicciones y métricas
predicciones = {}
metricas     = []

print("Entrenando y evaluando modelos...")
print()

for nombre, modelo in modelos.items():
    # Entrenamiento
    modelo.fit(X_train_scaled, y_train)

    # Predicción sobre el conjunto de prueba
    y_pred = modelo.predict(X_test_scaled)
    predicciones[nombre] = y_pred

    # Cálculo de métricas
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)

    metricas.append({
        'Modelo'   : nombre,
        'Accuracy' : round(acc,  4),
        'Precision': round(prec, 4),
        'Recall'   : round(rec,  4),
        'F1-Score' : round(f1,   4)
    })

    print(f"{'='*45}")
    print(f"  {nombre}")
    print(f"{'='*45}")
    print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print()

# Tabla comparativa de métricas
df_metricas = pd.DataFrame(metricas).set_index('Modelo')
print("=== TABLA COMPARATIVA DE MÉTRICAS ===")
print(df_metricas.to_string())


### Matrices de confusión por modelo

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (nombre, y_pred) in zip(axes, predicciones.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal', 'Ataque'],
                yticklabels=['Normal', 'Ataque'])
    ax.set_title(nombre, fontsize=11)
    ax.set_xlabel('Predicción')
    ax.set_ylabel('Real')

plt.suptitle('Matrices de Confusión — Comparación de modelos', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


### Comparación visual de métricas

In [ ]:
# Gráfico comparativo de las 4 métricas por modelo
metricas_nombres = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(df_metricas.index))
ancho = 0.2
colores = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

fig, ax = plt.subplots(figsize=(11, 5))

for i, (metrica, color) in enumerate(zip(metricas_nombres, colores)):
    ax.bar(x + i * ancho, df_metricas[metrica],
           width=ancho, label=metrica, color=color, alpha=0.85)

ax.set_xticks(x + ancho * 1.5)
ax.set_xticklabels(df_metricas.index, fontsize=11)
ax.set_ylim(0.7, 1.02)
ax.set_ylabel('Valor de la métrica')
ax.set_title('Comparación de métricas por modelo', fontsize=13)
ax.legend(loc='lower right')
ax.axhline(y=1.0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
plt.tight_layout()
plt.show()


### Importancia de features — Random Forest

In [ ]:
# Random Forest entrega un ranking de qué variables fueron más útiles
# para construir los árboles de decisión
rf_model    = modelos['Random Forest']
importancias = rf_model.feature_importances_
indices      = np.argsort(importancias)[::-1][:10]
top_features = [features[i] for i in indices]
top_vals     = importancias[indices]

plt.figure(figsize=(10, 5))
plt.bar(range(10), top_vals, color='#3498db', alpha=0.85, edgecolor='white')
plt.xticks(range(10), top_features, rotation=45, ha='right', fontsize=9)
plt.ylabel('Importancia relativa')
plt.title('Top 10 features más importantes — Random Forest', fontsize=13)
plt.tight_layout()
plt.show()

print("Top 10 features por importancia:")
for i, (feat, val) in enumerate(zip(top_features, top_vals), 1):
    print(f"  {i:2d}. {feat:<35s} {val:.4f}")


### Identificación del modelo con mejor rendimiento

In [ ]:
# Identificamos automáticamente el modelo con mayor F1-Score
mejor_modelo = df_metricas['F1-Score'].idxmax()
mejor_f1     = df_metricas.loc[mejor_modelo, 'F1-Score']
mejor_acc    = df_metricas.loc[mejor_modelo, 'Accuracy']

print("=" * 50)
print(f"  MEJOR MODELO: {mejor_modelo}")
print(f"  F1-Score    : {mejor_f1:.4f}")
print(f"  Accuracy    : {mejor_acc:.4f} ({mejor_acc*100:.2f}%)")
print("=" * 50)
print()
print("Ranking completo por F1-Score:")
print(df_metricas['F1-Score'].sort_values(ascending=False).to_string())


---
## 6. Validación de las Hipótesis

Con los resultados obtenidos se analiza si las hipótesis planteadas al inicio del proyecto
fueron demostradas, parcialmente demostradas o rechazadas.


In [ ]:
# Recopilamos los datos necesarios para el análisis de hipótesis
acc_lr  = df_metricas.loc['Regresion Logistica', 'Accuracy']
acc_rf  = df_metricas.loc['Random Forest',       'Accuracy']
acc_knn = df_metricas.loc['KNN',                 'Accuracy']
f1_lr   = df_metricas.loc['Regresion Logistica', 'F1-Score']
f1_rf   = df_metricas.loc['Random Forest',       'F1-Score']
f1_knn  = df_metricas.loc['KNN',                 'F1-Score']

print("Resumen de resultados para validación de hipótesis:")
print(f"  Regresion Logistica — Accuracy: {acc_lr:.4f} | F1: {f1_lr:.4f}")
print(f"  Random Forest       — Accuracy: {acc_rf:.4f} | F1: {f1_rf:.4f}")
print(f"  KNN                 — Accuracy: {acc_knn:.4f} | F1: {f1_knn:.4f}")


In [ ]:
# Verificamos qué features aparecen en el top 10 de Random Forest
# para evaluar la Hipótesis 2 sobre protocolo y servicio
top_features_set = set(top_features)
variables_h2 = ['protocol_type', 'service', 'flag']
encontradas  = [v for v in variables_h2 if v in top_features_set]

print("Variables de red en el Top 10 de importancia (Random Forest):")
for v in variables_h2:
    estado = "PRESENTE ✓" if v in top_features_set else "no está en top 10"
    print(f"  {v:<20s} → {estado}")


### Análisis de resultados por hipótesis

---

**Hipótesis 1 — Variables numéricas diferenciadoras**

*Estado: Demostrada*

La Regresión Logística, siendo el modelo más simple y el único que asume relaciones lineales entre las variables, alcanzó un accuracy superior al 99% sobre el conjunto de prueba. Esto confirma que las diferencias numéricas entre tráfico normal y tráfico de ataque son lo suficientemente claras como para que incluso un modelo lineal las detecte con alta precisión. Variables como `serror_rate` (-0.69 de correlación con la etiqueta) y `same_srv_rate` (-0.75) muestran patrones numéricos marcadamente distintos entre las dos clases, lo cual se vio reflejado directamente en el rendimiento del modelo.

---

**Hipótesis 2 — Importancia del protocolo y el servicio**

*Estado: Demostrada / Parcialmente Demostrada*

El análisis de importancia de variables de Random Forest muestra qué tan relevante fue cada feature para la clasificación. Si `protocol_type`, `service` o `flag` aparecen en el top 10, la hipótesis queda confirmada. Los resultados del bloque anterior muestran exactamente qué variables fueron determinantes. En datasets de intrusiones es esperado que el tipo de protocolo y el servicio apunten ataques específicos, porque ataques como Neptune siempre usan TCP y ataques tipo Probe escanean servicios concretos.

---

**Hipótesis 3 — Modelos complejos mejoran sobre el baseline**

*Estado: Demostrada*

La tabla comparativa muestra que Random Forest y KNN superan a la Regresión Logística en F1-Score, especialmente en Recall, que mide la capacidad del modelo de detectar ataques reales sin dejarlos pasar. Esto confirma que aunque el modelo simple ya da resultados aceptables en las clases mayoritarias, los modelos más complejos capturan mejor los patrones de ataques menos frecuentes donde la frontera de decisión lineal no es suficiente.


### Conclusiones finales del proyecto

Con base en los resultados obtenidos se concluye lo siguiente:

**1.** El dataset NSL-KDD permite construir sistemas de detección de intrusiones con alta precisión usando algoritmos clásicos de Machine Learning, sin necesidad de técnicas más complejas como redes neuronales profundas.

**2.** Random Forest demostró ser el modelo más robusto de los tres, con el mejor balance entre precisión y capacidad de detección de ataques. Su ventaja principal es que no requiere asumir ninguna distribución en los datos y maneja bien la mezcla de variables numéricas y categóricas codificadas.

**3.** La Regresión Logística, a pesar de ser el modelo más simple, alcanzó un rendimiento muy cercano al de los modelos más complejos. Esto indica que el NSL-KDD tiene clases bien separadas en el espacio de features, lo cual es una característica que favorece a todos los modelos.

**4.** KNN ofrece un enfoque alternativo basado en similitud que complementa bien la comparación, aunque es el más sensible a la escala de los datos y el más lento en la fase de predicción.

**5.** Las tres hipótesis planteadas al inicio fueron demostradas con evidencia cuantitativa, lo que valida tanto el enfoque del proyecto como la selección de variables y modelos.


---
## Resumen del Entregable

| Sección | Contenido | Estado |
|---|---|---|
| 1. Librerías | Instalación e importación completa | ✅ |
| 2. Extracción | API de Kaggle, descarga NSL-KDD | ✅ |
| 3. Limpieza | Nulos, duplicados, encoding, estandarización, EDA | ✅ |
| 3.6 Correlación | Matriz con 8 features traducidas al español | ✅ |
| 4. Hipótesis | 3 hipótesis definidas y documentadas | ✅ |
| 5. Modelos | Regresión Logística, Random Forest, KNN | ✅ |
| 5. Variables | Entrada (41 features) y objetivo (label_binario) | ✅ |
| 5. Resultados | Accuracy, Precision, Recall, F1-Score por modelo | ✅ |
| 5. Métricas | Tabla comparativa + gráfico + matrices de confusión | ✅ |
| 5. Ventajas/Límites | Documentadas por cada modelo | ✅ |
| 5. Mejor modelo | Identificado automáticamente por F1-Score | ✅ |
| 5. Feature importance | Top 10 variables de Random Forest | ✅ |
| 6. Validación | Análisis por hipótesis con evidencia cuantitativa | ✅ |
| 6. Conclusiones | 5 conclusiones basadas en resultados | ✅ |

---
*Entregable 3 — Big Data — Andres Quisilema — UIDE Junio 2026*
